In [6]:
#Poker kit library
!pip install -q pokerkit pyarrow
import pandas as pd
import numpy as np

In [7]:
# ---- Which dataset to process ----
# 100NL (original):  FOLDER = 'ABS-2009-07-01_2009-07-23_100NLH_OBFU/1',  TAG = ''
# 200NL (replication): FOLDER = 'ABS-2009-07-01_2009-07-23_200NLH_OBFU/2', TAG = '200NL'
FOLDER = 'ABS-2009-07-01_2009-07-23_200NLH_OBFU/2'
TAG = '200NL'

SUFFIX = f'_{TAG}' if TAG else ''   # '' keeps the original names (bets.parquet, decisions.parquet)
print(f'processing {FOLDER} -> bets{SUFFIX}.parquet, decisions{SUFFIX}.parquet')

processing ABS-2009-07-01_2009-07-23_200NLH_OBFU/2 -> bets_200NL.parquet, decisions_200NL.parquet


In [8]:
!rm -rf repo handhq
!git clone --filter=blob:none --sparse --depth 1 https://github.com/uoftcprg/phh-dataset.git repo
%cd repo
!git sparse-checkout set data/handhq/{FOLDER}
%cd ..

Cloning into 'repo'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 156 (delta 8), reused 153 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 784.50 KiB | 851.00 KiB/s, done.
Resolving deltas: 100% (8/8), done.
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 0), reused 2 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), 3.26 KiB | 3.26 MiB/s, done.
/content/repo
remote: Enumerating objects: 315, done.
remote: Counting objects: 100% (315/315), done.
remote: Compressing objects: 100% (258/258), done.
remote: Total 315 (delta 284), reused 87 (delta 57), pack-reused 0 (from 0)
Receiving objects: 100% (315/315), 21.94 MiB | 3.98 MiB/s, done.
Resolving deltas: 100% (284/284), done.
Updating files: 100% (315/315), done.
/content


In [9]:
import glob, shutil, os
src = f'repo/data/handhq/{FOLDER}'
found = sorted(glob.glob(f'{src}/**/*.phhs', recursive=True))
print(len(found), 'files found')
os.makedirs('handhq', exist_ok=True)
for p in found:
    shutil.copy(p, 'handhq/' + os.path.relpath(p, src).replace(os.sep, '_'))
print(len(os.listdir('handhq')), 'files in handhq/')
DATA_DIR = 'handhq'

310 files found
310 files in handhq/


In [10]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = f'/content/drive/MyDrive/capstone/handhq{SUFFIX}'   # each dataset gets its own folder
!mkdir -p {DATA_DIR}
!cp handhq/* {DATA_DIR}/

Mounted at /content/drive


In [11]:
"""
CODED BY AI
Correct per-player net results, computed from the action stream.

The 'winnings` field in the HandHQ->PHH conversion is NOT usable:
  - absent for 100% of showdown hands (~15% of hands, ~60% of money)
  - where present it is "amount collected", not net, and does not
    reconcile to zero-sum (tested: only 263/853 hands balance)

This module ignores it and reconstructs outcomes directly:
  - folded players lose exactly what they contributed
  - non-showdown: the one live player takes the pot
  - showdown: hands evaluated with pokerkit, side pots handled by layer

Guaranteed zero-sum by construction.
"""

import re
import tomllib
from pathlib import Path

try:
    from pokerkit import StandardHighHand
    HAS_EVAL = True
except ImportError:
    HAS_EVAL = False


def _split_cards(s):
    """'Kc7s4h' -> ['Kc','7s','4h']"""
    return [s[i:i + 2] for i in range(0, len(s), 2)]


def replay(hand):
    """
    Returns dict with contributions, board, revealed cards, fold status.
    """
    n = len(hand["starting_stacks"])
    antes = hand.get("antes", [])
    blinds = hand.get("blinds_or_straddles", [])

    contrib = [0.0] * n
    street_commit = [0.0] * n
    folded = [False] * n
    revealed = {}
    board = []

    for i in range(n):
        a = antes[i] if i < len(antes) else 0
        b = blinds[i] if i < len(blinds) else 0
        contrib[i] += (a or 0) + (b or 0)
        street_commit[i] = (b or 0)

    for act in hand.get("actions", []):
        t = act.split()
        if t[0] == "d":
            if t[1] == "db":
                board.extend(_split_cards(t[2]))
                street_commit = [0.0] * n
            continue
        m = re.match(r"^p(\d+)$", t[0])
        if not m:
            continue
        p = int(m.group(1)) - 1
        verb = t[1]
        if verb == "f":
            folded[p] = True
        elif verb == "cc":
            cb = max(street_commit)
            pay = cb - street_commit[p]
            contrib[p] += pay
            street_commit[p] = cb
        elif verb == "cbr":
            amt = float(t[2])
            contrib[p] += amt - street_commit[p]
            street_commit[p] = amt
        elif verb == "sm":
            if len(t) > 2 and "?" not in t[2]:
                revealed[p] = _split_cards(t[2])

    return {
        "n": n,
        "contrib": contrib,
        "folded": folded,
        "revealed": revealed,
        "board": board,
    }


def _rank(hole, board):
    """Higher is better. None if not evaluable."""
    if not HAS_EVAL or len(hole) != 2 or len(board) < 3:
        return None
    try:
        return StandardHighHand.from_game("".join(hole), "".join(board))
    except Exception:
        return None


def resolve(hand):
    """
    Returns (net_per_player, status).

    status: 'fold'      - won without showdown, exact
            'showdown'  - evaluated with pokerkit, exact
            'unknown'   - could not resolve (cards not revealed)
    """
    st = replay(hand)
    n, contrib, folded = st["n"], st["contrib"], st["folded"]
    live = [i for i in range(n) if not folded[i]]

    # --- build side-pot layers -------------------------------------
    levels = sorted(set(c for c in contrib if c > 0))
    layers = []   # (amount, eligible_live_players)
    prev = 0.0
    for lv in levels:
        amount = sum(min(c, lv) - min(c, prev) for c in contrib)
        elig = [i for i in live if contrib[i] >= lv]
        if amount > 1e-9:
            layers.append((amount, elig))
        prev = lv

    net = [-c for c in contrib]          # everyone starts down their contribution
    status = "fold"

    if len(live) == 1:
        for amount, elig in layers:
            # uncalled money returns to its owner if nobody else is eligible
            winners = elig if elig else [max(range(n), key=lambda i: contrib[i])]
            for w in winners:
                net[w] += amount / len(winners)
    else:
        # A player who mucks at showdown ('sm ????' or no sm) forfeits the
        # pot, so only players who actually revealed can win. This is a
        # standard poker rule and resolves the common "one shows, one mucks"
        # case without needing the mucker's cards.
        status = "showdown"
        ranks = {}
        for i in live:
            if i in st["revealed"]:
                r = _rank(st["revealed"][i], st["board"])
                if r is not None:
                    ranks[i] = r

        if not ranks:
            return [None] * n, "unknown"
        if len(ranks) == 1:
            status = "showdown_muck"   # winner by forfeit, still exact

        for amount, elig in layers:
            cand = [i for i in elig if i in ranks]
            if not cand:
                # nobody eligible revealed -> money returns to its owner
                owner = max(range(n), key=lambda i: contrib[i])
                net[owner] += amount
                continue
            best = max(ranks[i] for i in cand)
            winners = [i for i in cand if ranks[i] == best]
            for w in winners:
                net[w] += amount / len(winners)

    return net, status


def resolve_file(path):
    d = tomllib.load(open(path, "rb"))
    hands = [d[k] for k in sorted(d, key=int)] if "actions" not in d else [d]
    out = []
    for h in hands:
        net, status = resolve(h)
        out.append({"hand": h.get("hand"), "net": net,
                    "status": status, "players": h.get("players")})
    return out

In [12]:
"""
CODED BY AI
PHH/PHHS parser -> per-decision feature rows.

Emits one row per voluntary betting decision (cc / cbr / f), with the
game state as it stood BEFORE the decision was made.

Target variable for the bet-sizing models: bet_to_bb (and bet_to_pot).
"""

import tomllib
import re
from pathlib import Path

STREETS = ["preflop", "flop", "turn", "river"]


def load_phhs(path):
    """Load a .phh/.phhs file. Returns list of hand dicts."""
    with open(path, "rb") as f:
        data = tomllib.load(f)
    # Multi-hand files: keys are '1','2',... Single-hand files: fields at top level.
    if "actions" in data:
        return [data]
    # sort numerically so hand order is preserved
    return [data[k] for k in sorted(data.keys(), key=lambda x: int(x))]


def parse_hand(hand, source_file=""):
    """
    Replay one hand's action stream, emitting a row per player decision.

    Returns (rows, hand_meta).
    """
    players = hand.get("players")
    stacks0 = hand.get("starting_stacks", [])
    blinds = hand.get("blinds_or_straddles", [])
    antes = hand.get("antes", [])
    n = len(stacks0)

    if players is None:
        players = [f"seat{i+1}" for i in range(n)]

    bb = max(blinds) if blinds else None
    if not bb:
        return [], None

    # --- hand-level metadata (HandHQ files carry these; ACPC files do not) ---
    meta = {
        "source_file": source_file,
        "hand_id": hand.get("hand"),
        "venue": hand.get("venue"),
        "table": hand.get("table"),
        "day": hand.get("day"),
        "month": hand.get("month"),
        "year": hand.get("year"),
        "time": hand.get("time"),
        "bb": bb,
        "n_players": n,
        "has_timestamp": hand.get("time") is not None,
    }

    # --- state ---
    committed = [0.0] * n        # total chips put in this hand, per player
    street_commit = [0.0] * n    # chips put in on the current street
    stacks = list(stacks0)
    folded = [False] * n
    street = 0
    pot = 0.0

    # post antes and blinds
    for i in range(n):
        a = antes[i] if i < len(antes) else 0
        b = blinds[i] if i < len(blinds) else 0
        post = (a or 0) + (b or 0)
        committed[i] += post
        street_commit[i] += (b or 0)   # antes are not "live" bets to match
        stacks[i] -= post
        pot += post

    current_bet = max(street_commit) if street_commit else 0.0
    n_active = n
    action_idx = 0
    aggressor_this_street = None

    rows = []

    for act in hand.get("actions", []):
        tok = act.split()

        # dealer actions -------------------------------------------------
        if tok[0] == "d":
            if tok[1] == "db":           # board card(s) -> new street
                street += 1
                street_commit = [0.0] * n
                current_bet = 0.0
                aggressor_this_street = None
            continue                      # 'dh' (hole cards) carries no decision

        # player actions -------------------------------------------------
        m = re.match(r"^p(\d+)$", tok[0])
        if not m:
            continue
        p = int(m.group(1)) - 1          # 0-indexed
        verb = tok[1]

        if verb == "sm":                  # showdown reveal, not a decision
            continue

        to_call = max(0.0, current_bet - street_commit[p])
        pot_before = pot

        row = {
            **meta,
            "player": players[p] if p < len(players) else f"seat{p+1}",
            "seat_idx": p,
            "street": STREETS[street] if street < len(STREETS) else f"street{street}",
            "street_idx": street,
            "action_idx": action_idx,
            "pot_before": pot_before,
            "stack_before": stacks[p],
            "to_call": to_call,
            "n_active": n_active,
            "is_facing_bet": to_call > 0,
            "committed_before": committed[p],
            # normalized versions (comparable across stakes)
            "pot_before_bb": pot_before / bb,
            "stack_before_bb": stacks[p] / bb,
            "to_call_bb": to_call / bb,
            "spr": (stacks[p] / pot_before) if pot_before > 0 else None,
        }

        if verb == "f":
            row["action"] = "fold"
            row["bet_to"] = None
            folded[p] = True
            n_active -= 1

        elif verb == "cc":
            row["action"] = "call" if to_call > 0 else "check"
            row["bet_to"] = None
            pay = min(to_call, stacks[p])
            committed[p] += pay
            street_commit[p] += pay
            stacks[p] -= pay
            pot += pay

        elif verb == "cbr":
            amt = float(tok[2])           # 'cbr X' = raise TO X on this street
            row["action"] = "raise" if to_call > 0 else "bet"
            row["bet_to"] = amt
            row["bet_to_bb"] = amt / bb
            row["bet_to_pot"] = amt / pot_before if pot_before > 0 else None
            row["raise_increment"] = amt - current_bet
            pay = amt - street_commit[p]
            committed[p] += pay
            street_commit[p] += pay
            stacks[p] -= pay
            pot += pay
            current_bet = amt
            aggressor_this_street = p

        else:
            continue

        rows.append(row)
        action_idx += 1

    # Net result per player, reconstructed from the action stream.
    # The `winnings` field is deliberately NOT used: it is absent for all
    # showdown hands and does not reconcile to zero-sum where present.
    net, status = resolve(hand)

    for r in rows:
        i = r["seat_idx"]
        r["hand_net"] = net[i] if i < len(net) else None
        r["hand_net_bb"] = (r["hand_net"] / bb) if r["hand_net"] is not None else None
        r["outcome_status"] = status
        r["invested_total"] = committed[i]
        r["invested_total_bb"] = committed[i] / bb
        r["won_hand"] = (r["hand_net"] > 0) if r["hand_net"] is not None else None

    return rows, meta


def parse_file(path):
    hands = load_phhs(path)
    all_rows = []
    for h in hands:
        rows, _ = parse_hand(h, source_file=Path(path).name)
        all_rows.extend(rows)
    return all_rows

In [13]:
#Time stamp is important to track players change in decison making over time
def build_timestamp(df):
  ts = pd.to_datetime(
      dict(
          year=df["year"],
          month=df["month"],
          day=df["day"],
      ),
      errors="coerce", #marks missing data instead of crashing
  )
  tod = pd.to_timedelta(df["time"].astype(str), errors="coerce")
  return ts + tod

#30 min gap or new table dictates new session
def add_sessions(hands, gap_min=30):

  #create data table organized by player, table, timestamp
  d = hands.sort_values(["player", "table", "ts"]).copy()

  gap = d.groupby(["player", "table"])["ts"].diff() #get time gap between each hand played at a table
  new_session = (gap > pd.Timedelta(minutes=gap_min)) | gap.isna()
  d["session_id"] = new_session.groupby([d["player"], d["table"]]).cumsum()

  #Creates a unique id for a players session using player id, table, and session id num
  d["session_key"] = (
      d["player"].astype(str) + "|" + d["table"].astype(str) + "|" + d["session_id"].astype(str)
  )
  return d

#records loss streak
def streak(s):
    out, run = [], 0
    for v in s:
        out.append(run)
        run = run + 1 if v < 0 else 0
    return pd.Series(out, index=s.index)

#adds features to track useful info
def add_sequence_features(d):
  g = d.groupby("session_key", sort=False)

  d["hand_in_session"] = g.cumcount()
  d["session_elapsed_min"] = (d["ts"] - g["ts"].transform("min")).dt.total_seconds() / 60

  #track results of previous hand
  d["prev_net_bb"] = g["hand_net_bb"].shift(1)
  d["prev_was_loss"] = d["prev_net_bb"] < 0

  #Cumulative losses and loss streak
  d["cum_net_bb"] = g["hand_net_bb"].cumsum() - d["hand_net_bb"]
  d["loss_streak"] = g["hand_net_bb"].transform(streak)

  #Calculates big losses using std deviation
  #AI CODED
  pstd = d.groupby("player")["hand_net_bb"].transform("std")
  d["prev_loss_z"] = d["prev_net_bb"] / pstd
  d["prev_big_loss"] = d["prev_loss_z"] < -2

  #Record time taken to play as this can be an indicator of abnormal play
  d["secs_since_prev_hand"] = g["ts"].diff().dt.total_seconds()
  return d

#orders players by useful data like number of hands, sessions, and bets
def player_volume_report(d):
  v = (
      d.groupby("player").agg(
        hands=("hand_id", "nunique"),
        sessions=("session_key", "nunique"),
        bets=("bet_to_bb", "count"),
      ).sort_values("hands", ascending=False)
  )
  return v

In [14]:
"""
CODED BY AI
Concatenates all data files
"""
from pathlib import Path
files_ = sorted(Path(DATA_DIR).rglob('*.phhs')) + sorted(Path(DATA_DIR).rglob('*.phh'))
print(f'{len(files_)} files')

frames, per_file_players = [], {}
for i, f in enumerate(files_, 1):
    try:
        rows = parse_file(f)
    except Exception as e:
        print(f'  [skip] {f.name}: {e}'); continue
    if not rows: continue
    d = pd.DataFrame(rows)
    frames.append(d)
    per_file_players[f.name] = set(d.player.unique())
    if i % 25 == 0 or i == len(files_): print(f'  parsed {i}/{len(files_)}')

df = pd.concat(frames, ignore_index=True)

df['ts'] = build_timestamp(df)
df = df[df.hand_net.notna()]
print(f'\n{len(df):,} decision rows | {df.hand_id.nunique():,} hands | {df.player.nunique():,} players')

310 files
  parsed 25/310
  parsed 50/310
  parsed 75/310
  parsed 100/310
  parsed 125/310
  parsed 150/310
  parsed 175/310
  parsed 200/310
  parsed 225/310
  parsed 250/310
  parsed 275/310
  parsed 300/310
  parsed 310/310

2,624,719 decision rows | 309,176 hands | 5,505 players


In [15]:
#Overall reults per player per hand
hands = (df.sort_values('ts')
        .groupby(['player','hand_id'], as_index=False)
        .agg(table=('table','first'), ts=('ts','first'),
        hand_net_bb=('hand_net_bb','first')))
hands = add_sessions(hands, gap_min=30)
hands = add_sequence_features(hands)
print("Unique Sessions: ", hands.session_key.nunique())

Unique Sessions:  32468


In [16]:
#model_df records all individual decisions
#bets_df records all bet decisions
keep = ['player','hand_id','session_key','hand_in_session','prev_net_bb',
        'prev_was_loss','prev_big_loss','cum_net_bb','loss_streak',
        'secs_since_prev_hand','session_elapsed_min']
model_df = df.merge(hands[keep], on=['player','hand_id'], how='left')
bets_df = model_df[model_df.bet_to.notna()].copy()

model_df.to_parquet(f'decisions{SUFFIX}.parquet', index=False)
bets_df.to_parquet(f'bets{SUFFIX}.parquet', index=False)

In [17]:
#Save to drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/capstone
!cp decisions{SUFFIX}.parquet bets{SUFFIX}.parquet /content/drive/MyDrive/capstone/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
bets_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 485551 entries, 3 to 2624717
Data columns (total 47 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   source_file           485551 non-null  object        
 1   hand_id               485551 non-null  int64         
 2   venue                 485551 non-null  object        
 3   table                 485551 non-null  object        
 4   day                   485551 non-null  int64         
 5   month                 485551 non-null  int64         
 6   year                  485551 non-null  int64         
 7   time                  485551 non-null  object        
 8   bb                    485551 non-null  int64         
 9   n_players             485551 non-null  int64         
 10  has_timestamp         485551 non-null  bool          
 11  player                485551 non-null  object        
 12  seat_idx              485551 non-null  int64         
 13  str

## Data summary

### Output tables
| Table | One row = | Rows | Saved as | Used for |
|---|---|---|---|---|
| `df` | one decision (fold/check/call/bet/raise) | 2.93M | – | raw parsed data |
| `hands` | one player in one hand | ~2M | – | building sessions + loss history |
| `model_df` | one decision + loss history | 2.93M | `decisions.parquet` | all decisions |
| `bets_df` | one bet/raise + loss history | 518K | `bets.parquet` | **modelling table** |

Dataset (original run): HandHQ, Absolute Poker, $0.50/$1 No-Limit Hold'em (subfolder 1), July 2009.
325 files · 323,461 hands · 8,678 players · 38,193 sessions. The replication uses the $1/$2 (200NL) folder, set at the top.
All money columns ending in `_bb` are in big blinds so stakes are comparable.

### Columns in `bets_df`
**Role key:** 🎯 target (what the model predicts) · 🔵 model input · 🟠 validation only (NOT a model input) · ⚪ ID / metadata

#### Target
| Column | Role | Meaning |
|---|---|---|
| `bet_to_bb` | 🎯 | Size of the bet/raise, in big blinds (the total the player raised *to* on this street) |
| `bet_to_pot` | 🎯 | Same bet as a fraction of the pot before it (0.5 = half-pot) |
| `bet_to` | ⚪ | Raw bet size in dollars |
| `raise_increment` | ⚪ | How much above the previous bet this raise was, in dollars |

#### Game state before the decision (model inputs)
| Column | Role | Meaning |
|---|---|---|
| `street` / `street_idx` | 🔵 | preflop / flop / turn / river (0–3) |
| `pot_before_bb` | 🔵 | Pot size before this action |
| `stack_before_bb` | 🔵 | Player's remaining chips before this action |
| `to_call_bb` | 🔵 | Amount needed to call (0 = nobody has bet yet) |
| `is_facing_bet` | 🔵 | True if someone already bet this street |
| `spr` | 🔵 | Stack-to-pot ratio (stack ÷ pot) |
| `n_active` | 🔵 | Players still in the hand |
| `n_players` | 🔵 | Players dealt in |
| `seat_idx` | 🔵 | Position in the hand, starting from small blind (0 = SB, 1 = BB) |
| `action_idx` | ⚪ | How many decisions came before this one in the hand |
| `committed_before` | ⚪ | Chips this player already put in this hand, in dollars |
| `pot_before`, `stack_before`, `to_call` | ⚪ | Dollar versions of the columns above |
| `action` | ⚪ | "bet" (no prior bet) or "raise" (facing a bet) |

#### This hand's result (never a model input: it's decided *after* the bet)
| Column | Role | Meaning |
|---|---|---|
| `hand_net_bb` / `hand_net` | 🟠 | Player's profit/loss on this hand (rebuilt by replaying the hand; the dataset's own `winnings` field was unusable) |
| `won_hand` | 🟠 | True if profit > 0 |
| `invested_total_bb` / `invested_total` | 🟠 | Total chips the player put into this hand |
| `outcome_status` | ⚪ | How the hand ended: `fold`, `showdown`, `showdown_muck` |

#### Loss history before this hand (validation only, used to test the flags)
| Column | Role | Meaning |
|---|---|---|
| `prev_net_bb` | 🟠 | Result of the previous hand in this session (NaN on the first hand) |
| `prev_was_loss` | 🟠 | Previous hand was a loss (mostly just lost blinds; weak signal) |
| `prev_big_loss` | 🟠 | Previous loss was > 2 standard deviations below this player's normal result |
| `cum_net_bb` | 🟠 | Running session profit/loss before this hand |
| `loss_streak` | 🟠 | Consecutive losing hands right before this one |
| `hand_in_session` | 🟠 | Hands played so far this session (0 = first) |
| `session_elapsed_min` | 🟠 | Minutes since the session started |
| `secs_since_prev_hand` | 🟠 | Seconds since previous hand (pace of play) |

#### IDs and metadata
| Column | Meaning |
|---|---|
| `player` | Obfuscated player ID, consistent across files |
| `hand_id` | Site's hand number |
| `session_key` | `player|table|session#`; a session ends after a 30+ min break or a table change |
| `table`, `venue`, `source_file` | Where the hand came from |
| `ts` | Full timestamp (Eastern time) |
| `day`, `month`, `year`, `time`, `has_timestamp` | Raw date/time fields |
| `bb` | Big blind size in dollars ($1 here) |

### Why loss history is not a model input
The models learn a player's *normal* bet size from the game state alone. If loss history were an input, the model would learn "bets grow after losses" as normal behaviour and couldn't detect it. Instead, we flag bets that are far from the player's normal, then test whether those flags happen more often after big losses and losing streaks.